# Clustering in Real Life — The Complete Workflow (Wine Dataset)

This is how it's actually done: no hand-coding algorithms, just the right sklearn functions in the right order, and — most importantly — **knowing why** each step is there and **how to read the numbers**.

New dataset: **Wine** — 178 wines, 13 chemical measurements each, 3 real cultivars (grape types). 13 dimensions is too many to eyeball, which is exactly when these tools earn their keep.

**Your doubts, answered in this notebook:**
1. Do I *need* PCA before clustering, or can I cluster the standardized data directly?
2. What happens if I skip standardizing?
3. How do I evaluate clusters — what do all those numbers mean?
4. What's the actual real-life flow, start to finish?

---

In [15]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             adjusted_rand_score, confusion_matrix)

wine = load_wine()
X = wine.data
y_true = wine.target            # real cultivar — we PRETEND we don't have this, use only to check at the end
feature_names = wine.feature_names


df = pd.DataFrame(X, columns=feature_names)
print(f"{X.shape[0]} wines, {X.shape[1]} features, {len(set(y_true))} true cultivars")
df.head()

178 wines, 13 features, 3 true cultivars


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


## The problem you can SEE: features are on wildly different scales

This single fact drives the whole standardization question. Look at the raw ranges.

In [2]:
summary = pd.DataFrame({
    'mean': df.mean(),
    'min': df.min(),
    'max': df.max(),
    'range': df.max() - df.min()
}).round(2).sort_values('range', ascending=False)
print(summary)
print("\nLook at 'proline' (range ~1400) vs 'nonflavanoid_phenols' (range ~0.6).")
print("Proline's numbers are ~2000x bigger. Clustering uses distance —")
print("so proline would DOMINATE every distance calc just because of its units.")

                                mean     min      max    range
proline                       746.89  278.00  1680.00  1402.00
magnesium                      99.74   70.00   162.00    92.00
alcalinity_of_ash              19.49   10.60    30.00    19.40
color_intensity                 5.06    1.28    13.00    11.72
malic_acid                      2.34    0.74     5.80     5.06
flavanoids                      2.03    0.34     5.08     4.74
alcohol                        13.00   11.03    14.83     3.80
proanthocyanins                 1.59    0.41     3.58     3.17
total_phenols                   2.30    0.98     3.88     2.90
od280/od315_of_diluted_wines    2.61    1.27     4.00     2.73
ash                             2.37    1.36     3.23     1.87
hue                             0.96    0.48     1.71     1.23
nonflavanoid_phenols            0.36    0.13     0.66     0.53

Look at 'proline' (range ~1400) vs 'nonflavanoid_phenols' (range ~0.6).
Proline's numbers are ~2000x bigger. Clusteri

---

# DOUBT 1 & 2 — Do I need to standardize? Do I need PCA?

Let's not theorize — let's **run the experiment**. We'll cluster the same data 4 ways and score each against the true cultivars (ARI = 1.0 perfect, 0 = random). This is the most important cell in the notebook.

In [3]:
scores = {}

# Way 1: RAW data, straight into k-means (the naive approach)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
scores['1. raw (no prep)'] = adjusted_rand_score(y_true, km.labels_)

# Way 2: STANDARDIZE only, then k-means
X_std = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_std)
scores['2. standardized'] = adjusted_rand_score(y_true, km.labels_)

# Way 3: STANDARDIZE + PCA to 2D, then k-means
X_pca2 = PCA(n_components=2).fit_transform(X_std)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_pca2)
scores['3. std + PCA(2D)'] = adjusted_rand_score(y_true, km.labels_)

# Way 4: STANDARDIZE + PCA keeping 95% variance, then k-means
pca95 = PCA(n_components=0.95).fit(X_std)
X_pca95 = pca95.transform(X_std)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_pca95)
scores['4. std + PCA(95%)'] = adjusted_rand_score(y_true, km.labels_)

print(f"(PCA needed {X_pca95.shape[1]} components to keep 95% of variance)\n")
for name, s in scores.items():
    bar = '#' * int(s * 40)
    print(f"{name:20s} ARI={s:.3f}  {bar}")

(PCA needed 10 components to keep 95% of variance)

1. raw (no prep)     ARI=0.371  ##############
2. standardized      ARI=0.915  ####################################
3. std + PCA(2D)     ARI=0.895  ###################################
4. std + PCA(95%)    ARI=0.897  ###################################


In [4]:
fig = go.Figure(go.Bar(
    x=list(scores.values()), y=list(scores.keys()), orientation='h',
    marker_color=['#E24B4A', '#1D9E75', '#1D9E75', '#1D9E75'],
    text=[f'{v:.3f}' for v in scores.values()], textposition='outside'))
fig.update_layout(title='Standardizing is ESSENTIAL. PCA is OPTIONAL.',
    xaxis_title='ARI (agreement with true cultivars)', width=750, height=350,
    xaxis_range=[0, 1])
fig.show()

## The answer to your doubt, in one picture

- **Raw → standardized is a HUGE jump** (~0.37 → ~0.90). Skipping standardization let `proline` dominate and wrecked the clustering. **Standardizing is essential** whenever features have different units/scales (almost always).
- **PCA barely moved the score** (0.90 → 0.895 → 0.897). On this data PCA is **optional** — you can cluster the standardized data directly and get the same quality.

**So why does anyone use PCA before clustering? Three real reasons:**
1. **Visualization** — you can't plot 13D. PCA to 2D/3D lets you SEE the clusters (we'll do this next). Often the #1 reason.
2. **Speed** — with thousands of features (images, genes), 13→3 or 1000→50 makes clustering much faster.
3. **Noise reduction** — dropping tiny-variance directions can sometimes clean up the clustering.

If your data is already low-dimensional and clean (like Wine), you can skip PCA for the *clustering itself* — but you'll still want it to *visualize*. That's the nuance most tutorials skip.

> **Rule of thumb:** Always standardize. Use PCA when you need to see the data, when you have many features, or when speed matters. Don't use PCA as a reflex.

---

# The real-life flow, step by step

## Step 1 — Standardize (always)

In [5]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
print("After standardizing — every feature now mean~0, std~1:")
print(pd.DataFrame(X_std, columns=feature_names).describe().loc[['mean','std']].round(2).T.head())

After standardizing — every feature now mean~0, std~1:
                   mean  std
alcohol             0.0  1.0
malic_acid          0.0  1.0
ash                -0.0  1.0
alcalinity_of_ash  -0.0  1.0
magnesium          -0.0  1.0


## Step 2 — PCA to 2D, but ONLY to visualize

We'll cluster on the standardized 13D data (since PCA didn't help the score), but use PCA purely to make a 2D picture we can look at.

In [6]:
pca = PCA(n_components=2)
X_vis = pca.fit_transform(X_std)      # for plotting only
var_kept = pca.explained_variance_ratio_.sum()
print(f"2D view keeps {var_kept:.1%} of the variance")

fig = go.Figure(go.Scatter(x=X_vis[:,0], y=X_vis[:,1], mode='markers',
    marker=dict(size=8, color='#888', opacity=0.6)))
fig.update_layout(title='Wine in 2D (via PCA) — no labels. Can you see groups?',
    xaxis_title='PC1', yaxis_title='PC2', width=650, height=500)
fig.show()

2D view keeps 55.4% of the variance


## Step 3 — Choosing k WITHOUT knowing the answer

In real life you don't know there are 3 cultivars. Two standard tools to pick k:
- **Elbow method** — plot 'inertia' (total within-cluster spread) vs k. It always drops as k rises; look for the 'elbow' where it stops dropping fast.
- **Silhouette score** — measures how tight-and-separated clusters are (−1 to +1, higher better). Pick the k with the highest silhouette.

We cluster on the standardized 13D data here (the real clustering); the 2D plot was just for looking.

In [7]:
ks = range(2, 9)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_std)
    inertias.append(km.inertia_)                              # total within-cluster spread
    silhouettes.append(silhouette_score(X_std, km.labels_))  # tightness/separation

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Elbow: look for the bend', 'Silhouette: pick the peak'))
fig.add_trace(go.Scatter(x=list(ks), y=inertias, mode='lines+markers',
    marker_color='#534AB7'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(ks), y=silhouettes, mode='lines+markers',
    marker_color='#1D9E75'), row=1, col=2)
fig.update_xaxes(title_text='k', row=1, col=1)
fig.update_xaxes(title_text='k', row=1, col=2)
fig.update_yaxes(title_text='inertia (lower=tighter)', row=1, col=1)
fig.update_yaxes(title_text='silhouette (higher=better)', row=1, col=2)
fig.update_layout(title='Both methods point toward k=3 (silhouette peaks there)', width=900, height=400, showlegend=False)
fig.show()

best_k = list(ks)[int(np.argmax(silhouettes))]
print(f"Best k by silhouette: {best_k}")
for k, s in zip(ks, silhouettes):
    print(f"  k={k}: silhouette={s:.3f}")

Best k by silhouette: 3
  k=2: silhouette=0.268
  k=3: silhouette=0.286
  k=4: silhouette=0.252
  k=5: silhouette=0.232
  k=6: silhouette=0.240
  k=7: silhouette=0.198
  k=8: silhouette=0.133


**How to read these:**
- **Elbow:** inertia is the total squared distance from points to their center. It always falls as k grows (more centers = everyone closer to one). The 'elbow' — where the curve bends from steep to flat — suggests the natural k. Adding clusters past the elbow barely helps.
- **Silhouette:** for each point, compares how close it is to its own cluster vs the nearest other cluster. +1 = deep inside its cluster, 0 = on a boundary, −1 = probably misassigned. The average over all points is the silhouette score. Pick the k with the highest score.

These let you choose k with **no labels** — which is the real situation. (We happen to know it's 3, so it's a nice confirmation.)

## Step 4 — Run the final clustering with the chosen k

In [8]:
km = KMeans(n_clusters=best_k, n_init=10, random_state=0).fit(X_std)
labels = km.labels_

# plot the clusters on the 2D PCA view
colors = ['#534AB7', '#D85A30', '#1D9E75', '#D4537E', '#EF9F27']
fig = go.Figure()
for j in range(best_k):
    p = X_vis[labels == j]
    fig.add_trace(go.Scatter(x=p[:,0], y=p[:,1], mode='markers',
        marker=dict(size=8, color=colors[j], opacity=0.7), name=f'cluster {j}'))
fig.update_layout(title=f'Final k-means clustering (k={best_k}), shown on 2D PCA view',
    xaxis_title='PC1', yaxis_title='PC2', width=680, height=520)
fig.show()

---

# DOUBT 3 — Evaluation: what do all the numbers mean?

There are TWO situations, and they use different metrics:

**A) You have NO true labels (the normal case).** You can only judge clusters by their shape — are they tight and well-separated? Use *internal* metrics:
- **Silhouette score** (−1 to +1): higher = tighter, better-separated clusters.
- **Inertia / within-cluster sum of squares**: lower = tighter (but always drops with more k, so only compare same k).

**B) You DO have true labels (only for testing/learning).** You can directly measure agreement with the truth using *external* metrics:
- **ARI (adjusted Rand index)** (~0 to 1): 1 = perfect match, 0 = random. Adjusted so chance scores ~0.
- **Confusion matrix**: shows exactly which true class landed in which cluster.

Let's compute all of them.

In [9]:
# --- Internal metrics (no labels needed — what you'd use in real life) ---
sil = silhouette_score(X_std, labels)
print("INTERNAL metrics (judge shape, no labels):")
print(f"  silhouette score : {sil:.3f}   (>0.5 strong, 0.25-0.5 reasonable, <0.25 weak)")
print("   ~0.28 here = 'reasonable': the 3 clusters are real but two of them touch,")
print("   so border wines sit near the boundary. Matches what we see in the 2D plot.")
print(f"  inertia          : {km.inertia_:.1f}  (only meaningful compared across k)")

# --- External metrics (need true labels — only because this is a teaching set) ---
ari = adjusted_rand_score(y_true, labels)
print("\nEXTERNAL metrics (compare to truth — only possible because we have y_true):")
print(f"  ARI              : {ari:.3f}   (1=perfect, 0=random)")

INTERNAL metrics (judge shape, no labels):
  silhouette score : 0.286   (>0.5 strong, 0.25-0.5 reasonable, <0.25 weak)
   ~0.28 here = 'reasonable': the 3 clusters are real but two of them touch,
   so border wines sit near the boundary. Matches what we see in the 2D plot.
  inertia          : 1278.8  (only meaningful compared across k)

EXTERNAL metrics (compare to truth — only possible because we have y_true):
  ARI              : 0.915   (1=perfect, 0=random)


In [10]:
# Confusion matrix: true cultivar (rows) vs cluster found (cols)
cm = confusion_matrix(y_true, labels)
fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
    labels=dict(x='cluster found', y='true cultivar', color='count'),
    x=[f'cluster {i}' for i in range(best_k)],
    y=[f'cultivar {i}' for i in range(3)])
fig.update_layout(title='Confusion matrix: each true cultivar maps cleanly to one cluster',
    width=550, height=450)
fig.show()
print("Each row (true cultivar) should pile into ONE column (cluster).")
print("Off-diagonal counts = wines put in the 'wrong' cluster.")
print("Cluster numbers are arbitrary — what matters is each row concentrating in one column.")

Each row (true cultivar) should pile into ONE column (cluster).
Off-diagonal counts = wines put in the 'wrong' cluster.
Cluster numbers are arbitrary — what matters is each row concentrating in one column.


**Reading the silhouette per-point** — the plot below shows every wine's individual silhouette, grouped by cluster. Wide, tall bands all above the average line = healthy clusters. Points with negative silhouette (sticking left past 0) are likely misassigned.

In [11]:
sample_sil = silhouette_samples(X_std, labels)
fig = go.Figure()
y_lower = 0
for j in range(best_k):
    vals = np.sort(sample_sil[labels == j])
    y_upper = y_lower + len(vals)
    fig.add_trace(go.Scatter(
        x=vals, y=np.arange(y_lower, y_upper), mode='lines',
        fill='tozerox', line=dict(color=colors[j]), name=f'cluster {j}'))
    y_lower = y_upper
fig.add_vline(x=sil, line_dash='dash', line_color='red',
    annotation_text=f'avg = {sil:.2f}')
fig.update_layout(title='Silhouette per wine (grouped by cluster). Bands above the dashed avg = good.',
    xaxis_title='silhouette value', yaxis_title='wines (sorted within cluster)',
    width=720, height=480)
fig.show()

---

# Bonus — comparing all 3 algorithms the real way

In practice you'd try a few methods and compare their internal scores (and external, if you can). One clean table.

In [12]:
models = {
    'KMeans':  KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_std).labels_,
    'GMM':     GaussianMixture(n_components=3, random_state=0).fit(X_std).predict(X_std),
    'DBSCAN':  DBSCAN(eps=2.5, min_samples=5).fit(X_std).labels_,
}

rows = []
for name, lab in models.items():
    n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
    n_noise = (lab == -1).sum()
    # silhouette needs >1 cluster and no all-noise; guard it
    if n_clusters > 1:
        mask = lab != -1
        sil = silhouette_score(X_std[mask], lab[mask]) if mask.sum() > n_clusters else np.nan
    else:
        sil = np.nan
    ari = adjusted_rand_score(y_true, lab)
    rows.append([name, n_clusters, n_noise, round(sil,3) if sil==sil else 'n/a', round(ari,3)])

comp = pd.DataFrame(rows, columns=['method', 'clusters found', 'noise pts', 'silhouette', 'ARI vs truth'])
print(comp.to_string(index=False))
print("\nKMeans/GMM nail the 3 cultivars. DBSCAN struggles here because the")
print("clusters touch with no density gap — wrong tool for THIS data (eps is finicky).")

method  clusters found  noise pts silhouette  ARI vs truth
KMeans               3          0      0.286         0.915
   GMM               3          0      0.284         0.880
DBSCAN               1         24        n/a         0.005

KMeans/GMM nail the 3 cultivars. DBSCAN struggles here because the
clusters touch with no density gap — wrong tool for THIS data (eps is finicky).


---

# Your complete real-life cheat sheet

**The flow:**
```
1. Standardize          ALWAYS (StandardScaler) — features have different units
2. PCA                  OPTIONAL — for visualization, speed, or many features
3. Choose k             elbow + silhouette (when you don't know k)
4. Cluster              KMeans / GMM / DBSCAN
5. Evaluate             silhouette (no labels) or ARI + confusion matrix (with labels)
```

**Your doubts, settled:**
- *Do I need PCA before clustering?* No — standardized data clusters fine. PCA is for seeing the data, speed, or huge feature counts. It rarely changes the cluster quality on clean low-D data.
- *What if I skip standardizing?* Disaster (ARI 0.37 vs 0.90 here). Big-scale features hijack the distance math. Always standardize.
- *How do I evaluate?* No labels → silhouette (tightness/separation, higher better). Have labels → ARI (agreement, 1=perfect) + confusion matrix (which class went where).

**Reading the numbers:**
- silhouette > 0.5 strong, 0.25–0.5 okay, < 0.25 weak/overlapping
- ARI: 1 perfect, ~0 random (negative = worse than random)
- inertia: only compare across k (always shrinks with more k)
- confusion matrix: each true row should concentrate in one cluster column

**Next:** kernel methods + kernel PCA (clustering curved shapes), then PageRank, linear regression, bootstrap.